# HOPE-CAD — Kaggle comparison: CADIC train/eval → preflight → pilot

Upload this notebook. Attach `hope_cad_v7_ae829c5_kaggle_source.zip` as an Input dataset
(both a preserved ZIP and Kaggle's automatically unpacked source tree are supported).
Enable **Internet** and a **GPU**, then **Save Version → Save & Run All** to run in
the background. Only physical GPU0 is used, even when Kaggle allocates two cards.

The scientific source is commit `ae829c515023348ca12f1de84d35a342e8eaffaa`, all178
Git-blob hashes identical to the RTX3090 host source. Pixi installs the same locked
Linux Python3.13/PyTorch2.11+cu128 environment. No Kaggle-preinstalled torch, AMP,
TF32, batch bank updates, preprocessing changes, learner changes, or extra arms.
K2500/K5000 are independent full3629-image CADIC trainings, followed by official
1725-image final evaluation for each K. Research uses its separate fresh nine-arm
256-normal pilot; diagnostic/test results do not select its parameters.

**One-pass operator policy:** the user requested an unattended engineering run.
Running all cells authorizes the explicitly declared *automated technical review*
after preflight: independently verify source/runtime/manifest, every exported
payload and preflight checks, record its audit and operator attestation, then call
the unchanged review CLI to open **pilot only**. This is not an independent human
review, pilot acceptance, H32 adoption, or permission to start main/extension.
The final pilot always remains pending a human review. Set the policy flag False
below if you want the notebook to stop at preflight instead.

The driver has a10.5h total soft budget (including setup/diagnostic/evaluation),
leaving time before Kaggle's documented12h GPU-session limit. Finishing all nine
arms is **not guaranteed**, especially on T4. Atomic pauses continue in the same
session while time remains. Two verified rolling ZIP slots preserve all accumulated
raw exports/checkpoints in `/kaggle/working/hope_cad_stage1/`, without saving the
large Pixi environment/data. On timeout/failure they contain partial evidence,
never a false PASS. Same source/runtime/data remain required for any later resume.

[Kaggle notebooks](https://www.kaggle.com/docs/notebooks) ·
[Background execution](https://www.kaggle.com/page/GPU-tips-and-tricks) ·
[Pixi environment/cache locations](https://pixi.prefix.dev/latest/reference/environment_variables/)


In [ ]:
# Optional paths: leave None for automatic input discovery / pinned downloads.
SOURCE_INPUT = None       # ZIP path OR directory with bundle_manifest.json
MVTEC_ARCHIVE = None      # Optional exact pinned .tar.xz; otherwise assets downloads it
VIT_INPUT = None          # Optional exact .safetensors; otherwise assets downloads it
RESUME_SNAPSHOT = None    # Explicit previous stage1_snapshot_A/B.zip; default FRESH run

AUTO_TECHNICAL_PREFLIGHT_ATTESTATION = True  # preauthorized engineering policy above
SESSION_WALL_HOURS = 10.5
EXPORT_RESERVE_SECONDS = 600
RESEARCH_ID = 'hq32_kaggle_01'
DIAGNOSTIC_ID = 'cadic_check_01'
PIXI_VERSION = '0.83.0'
SOURCE_COMMIT = 'ae829c515023348ca12f1de84d35a342e8eaffaa'
SOURCE_DIGEST = '010d965704ec5970e235f764cb7c21e5640c3cd09361f72b6e37da4b3ab1414f'
SOURCE_ZIP_NAME = 'hope_cad_v7_ae829c5_kaggle_source.zip'
SOURCE_ZIP_SHA256 = '662a70716b2b236d01cf820274a23722174644a37efdff9e66e02565c91beb83'
MVTEC_SHA256 = 'cf4313b13603bec67abb49ca959488f7eedce2a9f7795ec54446c649ac98cd3d'
VIT_SHA256 = '2c4839589adbbf701573dd155e1814f8adb145cd4d235bb86a93269369e111ec'
HOST_REFERENCE = dict(instance='57551760',GPU='NVIDIA GeForce RTX 3090',
    source_digest=SOURCE_DIGEST,pipeline_seconds={'2500':534.8590980459412,'5000':1475.795095347712},
    scope='Reviewed single host run; observed timing ratio, not isolated GPU speedup or cross-runtime numerical parity')


In [ ]:
# Notebook kernel uses stdlib only. Every scientific command uses locked Pixi Python.
import csv, hashlib, io, json, math, os, platform, re, shutil, stat, subprocess
import tarfile, time, urllib.request, zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone

STARTED = time.monotonic()
BASE = Path('/kaggle/tmp/hope_cad_stage1')
REPO = BASE / 'source'
OUTPUT = Path('/kaggle/working/hope_cad_stage1')
INPUT = Path('/kaggle/input')
PIXI = BASE / 'tools/pixi'
DIAG = REPO / f'workspace/experiments/{DIAGNOSTIC_ID}/run'
STUDY = REPO / f'workspace/experiments/{RESEARCH_ID}/run'
RECORDS = REPO / 'workspace/records'
GPU_NAME = None
CONFIG = dict(source_commit=SOURCE_COMMIT, source_digest=SOURCE_DIGEST,
    diagnostic_budgets=[2500,5000], diagnostic_train_images_per_K=3629,
    diagnostic_test_images_per_K=1725, research_pilot_images_per_arm=256,
    diagnostic_id=DIAGNOSTIC_ID, research_id=RESEARCH_ID, one_GPU=True,
    automatic_technical_preflight_attestation=AUTO_TECHNICAL_PREFLIGHT_ATTESTATION,
    independent_human_review=False, automatic_pilot_acceptance=False,
    session_wall_hours=SESSION_WALL_HOURS, export_reserve_seconds=EXPORT_RESERVE_SECONDS,
    pixi_version=PIXI_VERSION, host_diagnostic_reference=HOST_REFERENCE)
ENV = dict(os.environ, PIXI_HOME=str(BASE/'pixi-home'), PIXI_CACHE_DIR=str(BASE/'cache'),
    XDG_CACHE_HOME=str(BASE/'cache'), PYTHONUNBUFFERED='1', PYTHONDONTWRITEBYTECODE='1',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', CUDA_VISIBLE_DEVICES='0',
    OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
    PIXI_NO_TELEMETRY='1', DO_NOT_TRACK='1')
for inherited in ('PYTHONPATH','PYTHONHOME','VIRTUAL_ENV','CONDA_PREFIX'):
    ENV.pop(inherited,None)

class SoftPause(Exception): pass

def utc(): return datetime.now(timezone.utc).isoformat()
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                    ensure_ascii=False, allow_nan=False).encode()).hexdigest()
def file_sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()
def read_json(path): return json.loads(Path(path).read_text())
def put_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False)+'\n')
    tmp.replace(path)
def remaining():
    return SESSION_WALL_HOURS*3600-(time.monotonic()-STARTED)-EXPORT_RESERVE_SECONDS
def hours(requested, minimum=240):
    if remaining() < minimum:
        raise SoftPause('Total notebook soft deadline reached; preserving partial evidence')
    return min(requested, remaining()/3600)
def safe_name(name):
    p=PurePosixPath(name)
    if not name or p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name:
        raise ValueError('Unsafe archive member: '+name)
    return p
def source_target(name):
    safe_name(name)
    target=REPO/name
    if not target.resolve().is_relative_to(REPO.resolve()): raise ValueError('Source path escapes')
    return target
def immutable_copy_bytes(target, raw):
    if target.exists():
        if target.read_bytes()!=raw: raise ValueError('Refuse changing existing source/state: '+str(target))
        return
    target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(raw)
def checked_source(manifest, getter):
    hashes=manifest['files']
    if (manifest.get('schema')!='hope_cad_portable_source_bundle_v1'
            or manifest['origin']['source_commit']!=SOURCE_COMMIT
            or digest(hashes)!=SOURCE_DIGEST or len(hashes)!=178):
        raise ValueError('Wrong scientific source bundle')
    for name, expected in hashes.items():
        safe_name(name); raw=getter(name)
        if hashlib.sha256(raw).hexdigest()!=expected: raise ValueError('Source checksum: '+name)
        immutable_copy_bytes(source_target(name), raw)
    immutable_copy_bytes(REPO/'bundle_manifest.json', json.dumps(manifest,sort_keys=True).encode())
    return hashes
def choose_source():
    if SOURCE_INPUT: return Path(SOURCE_INPUT)
    candidates=sorted(INPUT.rglob(SOURCE_ZIP_NAME))
    if len(candidates)==1: return candidates[0]
    if candidates: raise ValueError('Several source ZIPs; set SOURCE_INPUT explicitly')
    trees=[]
    for p in INPUT.rglob('bundle_manifest.json'):
        try:
            m=read_json(p)
            if m.get('origin',{}).get('source_commit')==SOURCE_COMMIT and digest(m.get('files',{}))==SOURCE_DIGEST:
                trees.append(p.parent)
        except (OSError, ValueError, KeyError): pass
    if len(trees)!=1:
        raise FileNotFoundError('Attach the provided source ZIP as Kaggle Input; ambiguous/missing source')
    return trees[0]
def extract_source():
    path=choose_source()
    if path.is_file():
        if file_sha(path)!=SOURCE_ZIP_SHA256: raise ValueError('Wrong source ZIP SHA256')
        with zipfile.ZipFile(path) as z:
            names=z.namelist()
            if len(names)!=len(set(names)): raise ValueError('Duplicate source entries')
            for n in names: safe_name(n)
            m=json.loads(z.read('bundle_manifest.json'))
            if set(names)!=set(m['files'])|{'bundle_manifest.json'}: raise ValueError('Source inventory mismatch')
            checked_source(m,z.read)
    else:
        m=read_json(path/'bundle_manifest.json')
        def original(name):
            p=path/name
            if p.is_symlink() or not p.resolve().is_relative_to(path.resolve()): raise ValueError('Source symlink')
            return p.read_bytes()
        checked_source(m,original)
    print('Verified178 source files; exact host source digest:', SOURCE_DIGEST, flush=True)
def run_process(args, label, cwd=None):
    OUTPUT.mkdir(parents=True, exist_ok=True)
    logdir=OUTPUT/'logs'; logdir.mkdir(exist_ok=True)
    logfile=logdir/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+label+'_'+str(time.time_ns())+'.log')
    started=time.monotonic(); print('\n>>>', label, flush=True)
    with logfile.open('w') as log:
        p=subprocess.Popen([str(a) for a in args],cwd=cwd or REPO,env=ENV,
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in p.stdout:
                print(line,end='',flush=True); log.write(line); log.flush()
            code=p.wait()
        except BaseException:
            # Do not snapshot changing model state after a notebook interrupt.
            p.wait(); raise
    rec=dict(label=label, exit_code=code,wall_seconds=time.monotonic()-started,utc=utc(),log=str(logfile))
    journal=OUTPUT/'invocations.json'
    records=read_json(journal) if journal.exists() else []
    records.append(rec);put_json(journal, records)
    return code
def must(args,label):
    if run_process(args,label): raise RuntimeError(label+' failed; see output logs')
def pixi_python(code,label):
    must([PIXI,'run','--locked','python','-B','-c',code],label)
def science(args,label):
    return run_process(['bash','scripts/remote/run.sh',*args],label)
def verify_payload_zip(path, header='artifact_manifest.json'):
    with zipfile.ZipFile(path) as z:
        names=z.namelist()
        if len(names)!=len(set(names)): raise ValueError('Duplicate ZIP entries')
        for name in names: safe_name(name)
        m=json.loads(z.read(header))
        if set(names)!=set(m['files'])|{header}: raise ValueError('ZIP inventory differs')
        for name, expected in m['files'].items():
            with z.open(name) as f:
                if hashlib.file_digest(f,'sha256').hexdigest()!=expected: raise ValueError('ZIP checksum: '+name)
    return m
def restore_snapshot():
    if not RESUME_SNAPSHOT: return
    path=Path(RESUME_SNAPSHOT)
    m=verify_payload_zip(path,'snapshot_manifest.json')
    if m.get('schema')!='hope_kaggle_stage1_snapshot_v1' or m.get('source_digest')!=SOURCE_DIGEST:
        raise ValueError('Foreign snapshot; no migration/relabeling')
    with zipfile.ZipFile(path) as z:
        for name in m['files']:
            if not name.startswith('source/'): raise ValueError('Unexpected snapshot path')
            target=source_target(name.removeprefix('source/'))
            immutable_copy_bytes(target,z.read(name))
    for ledger in STUDY.glob('budget_*.json'):
        if any(v.get('status')!='closed_after_verified_export' for v in read_json(ledger)['attempts']):
            raise ValueError('Unresolved reservation in snapshot; reviewer recovery required')
    print('Restored verified snapshot; runtime/data identity still checked by original loaders',flush=True)
def publish_snapshot(reason):
    if not (REPO/'bundle_manifest.json').exists(): return
    for run in (DIAG,STUDY):
        if (run/'.writer.lock').exists(): raise RuntimeError('Active writer; refuse snapshot')
    payload={}
    for name in read_json(REPO/'bundle_manifest.json')['files']:
        payload['source/'+name]=REPO/name
    payload['source/bundle_manifest.json']=REPO/'bundle_manifest.json'
    for base in (DIAG.parent,STUDY.parent,RECORDS):
        if not base.exists(): continue
        for p in sorted(base.rglob('*')):
            if p.is_symlink(): raise ValueError('Symlink in snapshot')
            if p.is_file() and p.name!='.writer.lock' and not p.name.endswith(('.tmp','.import_tmp')):
                payload['source/'+p.relative_to(REPO).as_posix()]=p
    manifest=dict(schema='hope_kaggle_stage1_snapshot_v1',source_digest=SOURCE_DIGEST,
        source_commit=SOURCE_COMMIT,reason=reason,utc=utc(),manual_pilot_review_required=True,
        is_qualified_stage_handoff_v1=False,files={n:file_sha(p) for n,p in payload.items()})
    pointer=OUTPUT/'snapshot_latest.json'
    old=read_json(pointer) if pointer.exists() else None
    slot='B' if old and old['slot']=='A' else 'A'
    target=OUTPUT/f'stage1_snapshot_{slot}.zip';tmp=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=2) as z:
        for n,p in payload.items(): z.write(p,n)
        z.writestr('snapshot_manifest.json',json.dumps(manifest,sort_keys=True,allow_nan=False))
    verify_payload_zip(tmp,'snapshot_manifest.json'); tmp.replace(target)
    receipt=dict(slot=slot,path=target.name,sha256=file_sha(target),bytes=target.stat().st_size,reason=reason,utc=utc())
    put_json(pointer,receipt)
    (OUTPUT/'SHA256SUMS').write_text('\n'.join(file_sha(p)+'  '+p.name for p in sorted(OUTPUT.glob('*.zip')))+'\n')
    print('VERIFIED SNAPSHOT:',receipt,flush=True)


In [ ]:
def prepare():
    if platform.system()!='Linux' or not INPUT.is_dir():
        raise RuntimeError('Run this notebook on Kaggle Linux GPU; no local GPU work')
    if not 0 < SESSION_WALL_HOURS <= 11 or EXPORT_RESERVE_SECONDS < 600:
        raise ValueError('Keep notebook soft limit<=11h and export reserve>=600s')
    for value in (RESEARCH_ID,DIAGNOSTIC_ID):
        if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9_-]{0,79}',value): raise ValueError('Invalid experiment ID')
    if RESEARCH_ID==DIAGNOSTIC_ID: raise ValueError('Diagnostic/research must have separate runs')
    OUTPUT.mkdir(parents=True,exist_ok=True);BASE.mkdir(parents=True,exist_ok=True)
    extract_source();restore_snapshot()
    put_json(OUTPUT/'operator_policy.json',CONFIG)
    put_json(RECORDS/'kaggle_operator_policy.json',CONFIG)
    if not PIXI.exists():
        PIXI.parent.mkdir(parents=True,exist_ok=True)
        url=f'https://github.com/prefix-dev/pixi/releases/download/v{PIXI_VERSION}/pixi-x86_64-unknown-linux-musl.tar.gz'
        archive=BASE/'pixi.tar.gz';urllib.request.urlretrieve(url,archive)
        with tarfile.open(archive,'r:gz') as tf:
            members=[m for m in tf.getmembers() if m.isfile() and PurePosixPath(m.name).name=='pixi']
            if len(members)!=1: raise ValueError('Unexpected Pixi binary archive')
            PIXI.write_bytes(tf.extractfile(members[0]).read());PIXI.chmod(0o755)
    result=subprocess.check_output([str(PIXI),'--version'],env=ENV,text=True).strip()
    if result!='pixi '+PIXI_VERSION: raise ValueError('Unexpected Pixi version: '+result)
    ENV['PATH']=str(PIXI.parent)+os.pathsep+ENV.get('PATH','')
    put_json(OUTPUT/'pixi_install.json',dict(version=result,binary_sha256=file_sha(PIXI),utc=utc()))
    must([PIXI,'install','--locked'],'pixi_install')
    must([PIXI,'run','--locked','check-install'],'check_install')
    must([PIXI,'run','--locked','env-check','--gpu'],'environment_check')
    must([PIXI,'run','--locked','test'],'full_test_suite')
    # CPU threads, TF32/AMP policy and package/CUDA pins come from unchanged code.
    pixi_python(r"""
import json,os,torch,subprocess
from pathlib import Path
from hope_cad.runtime.identity import configure_runtime,check_remote_runtime,source_digest
configure_runtime()
assert source_digest()=='010d965704ec5970e235f764cb7c21e5640c3cd09361f72b6e37da4b3ab1414f'
name=torch.cuda.get_device_name(0);rt=check_remote_runtime('cuda:0',name)
cap=torch.cuda.get_device_capability(0)
assert 'sm_'+str(cap[0])+str(cap[1]) in torch.cuda.get_arch_list(), 'Wheel lacks GPU native architecture'
x=torch.ones(4,4,device='cuda:0');assert torch.equal(x@x,torch.full_like(x,4));torch.cuda.synchronize()
p=Path('workspace/records/kaggle_runtime.json');p.parent.mkdir(parents=True,exist_ok=True)
p.write_text(json.dumps(dict(runtime=rt,only_cuda0=True,all_devices=subprocess.check_output(
 ['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],text=True)),indent=2))
""",'locked_runtime_and_GPU')
    global GPU_NAME
    GPU_NAME=read_json(RECORDS/'kaggle_runtime.json')['runtime']['device_name']
    archive=Path(MVTEC_ARCHIVE) if MVTEC_ARCHIVE else None
    if archive is None:
        found=sorted(INPUT.rglob('mvtec_anomaly_detection.tar.xz'))
        if len(found)==1: archive=found[0]
    args=[PIXI,'run','--locked','assets']
    if archive:
        if file_sha(archive)!=MVTEC_SHA256: raise ValueError('MVTec archive differs from host pinned archive')
        args += ['--archive',archive,'--archive-sha256',MVTEC_SHA256]
    vit=Path(VIT_INPUT) if VIT_INPUT else None
    if vit is None:
        found=sorted(INPUT.rglob('vit_base_patch8_224.augreg_in21k.safetensors'))
        if len(found)==1: vit=found[0]
    if vit:
        if file_sha(vit)!=VIT_SHA256: raise ValueError('Wrong ViT SHA256')
        dest=REPO/'workspace/weights/vit_base_patch8_224.augreg_in21k.safetensors'
        dest.parent.mkdir(parents=True,exist_ok=True)
        if not dest.exists(): shutil.copyfile(vit,dest)
        elif file_sha(dest)!=VIT_SHA256: raise ValueError('Existing ViT differs')
    must(args,'assets')
    # No fictitious rental price for Kaggle: omit USD allocation; unchanged B cap=12h.
    # No learned state is transferred between diagnostic and research.
    pixi_python(r"""
import json
from pathlib import Path
from hope_cad.data.fulltrain import build_manifest
from hope_cad.runtime.identity import source_digest
m,d=build_manifest('workspace/data/mvtec',source_digest())
assert len(m['online'])==3629 and sum(r['task_id']==0 for r in m['online'])==209
assert d=='95b2093469bd75b71e913739c21f66115061322eb0708f4697fb49b17a0c04f4', 'Host normal data/order differs'
Path('workspace/records/kaggle_data_comparison.json').write_text(json.dumps(dict(
 normal_rows_digest=d,images=3629,bottle=209,host_normal_rows_exact=True),indent=2))
""",'host_data_identity')
    publish_snapshot('setup_complete')

def diagnostic_phase(phase):
    wanted='diagnostic_train_complete' if phase=='train' else 'diagnostic_final_evaluation_complete'
    # Always enter the original CLI, even for completed phases: it validates
    # source/runtime/data/checkpoint bindings and skips already-completed work.
    for _ in range(24):
        h=hours(1)
        code=science(['cadic-diagnostic','--phase',phase,'--run-dir',DIAG,
             '--expected-gpu',GPU_NAME,'--max-hours',str(h)],'cadic_'+phase)
        publish_snapshot('cadic_'+phase+'_invocation')
        receipt=read_json(DIAG/'handoff.json')
        if code: raise RuntimeError('CADIC '+phase+' failed; evidence preserved')
        if receipt['status']==wanted: return
        if receipt['status']!='paused_at_committed_boundary': raise RuntimeError('Unexpected diagnostic state')
    raise SoftPause('Diagnostic invocation safety bound reached')

def research_phase(phase):
    if phase=='preflight' and (STUDY/'preflight.json').exists(): return
    if phase=='pilot' and (STUDY/'pilot.json').exists():
        if read_json(STUDY/'pilot.json').get('passed') is not True: raise RuntimeError('Existing pilot gates failed')
        arms=('CADIC','R_EXACT','FAST_ONLY','BIAS_VETO','PILOT_RATE_RANDOM_VETO','R_FO','H32_EXACT','H32_FO','H32_DIRECT')
        for arm in arms:
            folder=STUDY/'pilot'/(arm+'_seed0')
            if read_json(folder/'train_complete.json')['images']!=256:
                raise ValueError('Incomplete pilot arm: '+arm)
            active=read_json(folder/'active.json')
            if active['cursor']!=256 or file_sha(folder/'active.pt')!=active['sha256']:
                raise ValueError('Pilot checkpoint mismatch: '+arm)
        return
    for invocation in range(24):
        h=hours(3 if phase=='preflight' or invocation>0 else 1)
        code=science(['research','--phase',phase,'--run-dir',STUDY,'--device','cuda:0',
             '--expected-gpu',GPU_NAME,'--session-hours',str(h),'--external-billed-seconds','0'],'research_'+phase)
        publish_snapshot('research_'+phase+'_invocation')
        receipt=read_json(STUDY/'handoff.json')
        if code: raise RuntimeError('Research '+phase+' failed; no guard bypass/retry/tuning')
        expected='research_preflight_passed_pending_review' if phase=='preflight' else 'pilot_complete_pending_review'
        if receipt['status']==expected: return
        if receipt['status']!='paused_at_atomic_boundary': raise RuntimeError('Research gates did not pass: '+receipt['status'])
    raise SoftPause('Research invocation safety bound reached')


In [ ]:
def technical_preflight_review():
    if not AUTO_TECHNICAL_PREFLIGHT_ATTESTATION:
        raise SoftPause('Preflight ready; operator chose manual review before pilot')
    # This reviewer is outside the scientific runner; it reads receipts/source/ZIPs
    # and does not run optimizer steps, retune, alter locks, or look at AD metrics.
    code=r"""
import hashlib,json,math,zipfile
from pathlib import Path
import torch
from hope_cad.contract import digest,file_sha
from hope_cad.runtime.identity import configure_runtime,source_digest,check_remote_runtime
from hope_cad.data.fulltrain import build_manifest
from hope_cad.runtime.research import binding
configure_runtime()
run=Path(RUN_DIR)
p=json.loads((run/'preflight.json').read_text())
rt=check_remote_runtime('cuda:0',torch.cuda.get_device_name(0))
m,rd=build_manifest('workspace/data/mvtec',source_digest())
ident=binding(rt,rd,file_sha('workspace/weights/vit_base_patch8_224.augreg_in21k.safetensors'))
assert p['status']=='research_preflight_passed_pending_review'
assert p['binding']==ident and p['binding_digest']==digest(ident)
assert json.loads((run/'suite_lock.json').read_text())['identity']==ident
assert json.loads((run/'manifest_train.json').read_text())==m
c=p['checks'];n=c['numerical'];b=c['full_budget_pair_parity']
assert n['status']=='passed' and n['historical_gradient_nonzero'] and n['exact_FO_forward_equal']
assert n['all_active_connected'] and n['grid']==[28,28] and n['channels']==32
assert math.isfinite(n['exact_loss'])
assert c['optimized_NN_Eq9_exact'] is True
for key in ('passed','pair_exact','decisions_and_counters_exact','features_exact',
            'pixel_and_image_scores_exact','main_state_rng_unchanged'):
    assert b[key] is True, key
assert b['bank_count']==10000 and b['candidate_vectors']==4
assert set(c['clones'])=={'CADIC','R_EXACT','H32_EXACT','H32_DIRECT'}
assert all(v['images']==24 and v['extra_resume_images']==4 for v in c['clones'].values())
sessions=sorted((run.parent/'exports').glob('session_*'))
latest=sessions[-1];archive=latest/'evidence.zip';acc=json.loads((latest/'accounting.json').read_text())
assert acc['archive']['sha256']==file_sha(archive)
assert acc['attempt']['status']=='closed_after_verified_export'
with zipfile.ZipFile(archive) as z:
    names=z.namelist();assert len(names)==len(set(names))
    a=json.loads(z.read('artifact_manifest.json'))
    assert set(names)==set(a['files'])|{'artifact_manifest.json'}
    for name,expected in a['files'].items():
        with z.open(name) as f:assert hashlib.file_digest(f,'sha256').hexdigest()==expected
    assert json.loads(z.read('preflight.json'))==p
ledger=json.loads((run/'budget_B.json').read_text())
assert all(v['status']=='closed_after_verified_export' for v in ledger['attempts'])
assert len(m['online'])==3629 and rd=='95b2093469bd75b71e913739c21f66115061322eb0708f4697fb49b17a0c04f4'
assert not (run/'pilot_review.json').exists(), 'No automatic pilot acceptance'
audit=dict(status='automated_technical_review_passed',independent_human_review=False,
 source_digest=source_digest(),preflight_receipt_sha256=file_sha(run/'preflight.json'),
 archive_sha256=file_sha(archive),purpose='User-preauthorized engineering pilot only; final human review pending',
 official_test_metrics_used_for_review=False,candidate_adopted=False)
Path('workspace/records/kaggle_preflight_technical_audit.json').write_text(json.dumps(audit,indent=2))
print(json.dumps(audit,indent=2))
""".replace('RUN_DIR',repr(str(STUDY)))
    pixi_python(code,'independent_receipt_and_technical_audit')
    # Explicitly preserve the kind of attestation; never call it human review.
    if not (STUDY/'preflight_review.json').exists():
        note=('Operator preauthorized unattended engineering pipeline. Automated receipt/hash/runtime/data '
              'technical review passed; not independent human review, pilot acceptance, or H32 adoption.')
        code=science(['research-review','--run-dir',STUDY,'--phase','preflight','--accept','--note',note],
                     'preauthorized_technical_preflight_attestation')
        if code: raise RuntimeError('Preflight attestation failed')
    publish_snapshot('technical_preflight_attestation_recorded')

def final_report(status,error=None):
    report=dict(status=status,error=error,utc=utc(),elapsed_notebook_seconds=time.monotonic()-STARTED,
        source_commit=SOURCE_COMMIT,source_digest=SOURCE_DIGEST,GPU=GPU_NAME,
        independent_human_review=False,pilot_review_pending=True,main_started=False,
        Kaggle_to_Vast_stage_migration_qualified=False)
    if (DIAG/'summary.json').exists(): report['diagnostic']=read_json(DIAG/'summary.json')
    if (STUDY/'pilot.json').exists(): report['pilot']=read_json(STUDY/'pilot.json')
    put_json(OUTPUT/'stage1_report.json',report)
    put_json(RECORDS/'kaggle_stage1_report.json',report)
    rows=[]
    for k in (2500,5000):
        folder=DIAG/f'K{k}';row=dict(K=k)
        if (folder/'train_complete.json').exists():
            train=read_json(folder/'train_complete.json')
            row.update({n:train[n] for n in ('images','input_seconds','features_seconds','update_seconds','pipeline_seconds')})
            row['reference_host_pipeline_seconds']=HOST_REFERENCE['pipeline_seconds'][str(k)]
            row['observed_host_time_over_this_time']=row['reference_host_pipeline_seconds']/row['pipeline_seconds']
        if (folder/'final_macro.json').exists(): row.update(read_json(folder/'final_macro.json'))
        rows.append(row)
    keys=sorted(set().union(*(r.keys() for r in rows)))
    with (OUTPUT/'cadic_comparison.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=keys);writer.writeheader();writer.writerows(rows)
    print('\nFINAL STATUS:',status,'; human pilot review still pending',flush=True)
    for row in rows: print(row,flush=True)
    return report

def run_all():
    status='failed_preserve_evidence';error=None
    try:
        prepare()
        diagnostic_phase('train')
        diagnostic_phase('evaluate')
        # Reading anomaly results never branches/selects the research recipe.
        research_phase('preflight')
        technical_preflight_review()
        research_phase('pilot')
        status='stage1_pilot_complete_pending_human_review'
    except SoftPause as exc:
        status='paused_preserved';error=str(exc)
    except Exception as exc:
        error=type(exc).__name__+': '+str(exc)
        print('STOPPING WITHOUT TUNING:',error,flush=True)
    finally:
        final_report(status,error)
        publish_snapshot(status)
    return read_json(OUTPUT/'stage1_report.json')


## Run the requested pipeline

This cell is the only launch point. It uses the original CLI and its failure,
runtime/data/source, budget and checkpoint guards; exceptions do not silently
continue to the next stage. No Vast credentials, Stop/Destroy, API keys, Git push,
main training, parameter search or anomaly-metric selection.

Compare **CADIC pipeline components, full invocation time including exports,
evaluation wall/metrics and per-arm pilot telemetry** with the same fields on
Vast. Setup/download/cache warmness, session splits/restore/export overhead,
CPU/GPU/driver and platform differ and are reported; a single run does not isolate
the cause of a speed difference. Peak memory of this diagnostic does not bound
H32/main memory. Equal recipe/source is not a promise of bitwise GPU parity.

At completion download the `stage1_report.json`, `cadic_comparison.csv`, logs and
`snapshot_latest.json` plus its named `stage1_snapshot_A/B.zip` and SHA256SUMS from
Notebook Output. A snapshot includes **all accumulated raw session exports,
accounting sidecars, live checkpoints, records and exact source**, but excludes
dataset/backbone/environment. The rolling slots replace only redundant outer
containers after full hash verification; raw session evidence remains inside the
latest snapshot. The ZIP is a Kaggle backup, **not** a qualified cross-host
`handoff-pack` archive. Do not infer Phase2 permission from it. Keep failed/partial
outputs too. Optional later resume must use the exact source and runtime/data;
attach one selected snapshot and set RESUME_SNAPSHOT explicitly, never relabel
Vast/T4 states or delete locks/ledgers to bypass a mismatch.


In [ ]:
result = run_all()
print(json.dumps({k:result[k] for k in ('status','error','GPU','elapsed_notebook_seconds')},indent=2))
print('Download outputs from:',OUTPUT)
